<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Découverte de données satellitaires - Productions Eumetsat - Format TIFF</div>

---

## 🎯 **Objectifs du TP**

##  🔎 Analyser une image satellitaire TIFF géoréférencée
##  🔄 Visualiser une image et la reprojeter

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise principalement la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **TIFF**, issus de la production opérationnelle disponible sur Eumetview.


### Qu'est-ce qu'ImageMagick ?
ImageMagick est une suite **logicielle libre et open source** dédiée à la **création**, la **conversion**, la **modification** et l'**analyse** d'**images raster**. C'est un outil très utilisé en automatisation, traitement d'images scientifiques, développement web et production graphique. </br>
Il permet de manipuler plus de 200 formats d'images, notamment : JPEG, PNG, TIFF, GIF, PDF, SVG... </br>
Il fonctionne principalement en ligne de commande, mais peut aussi être intégré dans des scripts ou des applications via des bibliothèques disponibles en C, C++, Python, PHP, Java, etc. </br>
https://imagemagick.org</br>

---

## 📂 **Fichier utilisé**

Le fichier TIFF étudié est un produit de type **GeoColor RGB** du satellite Meteosat-12 :

📁 `/stockage/DATA/eumetview/20261007_geocolor_rgb_eumetview.tif`

> 💡 *Ce fichier est un produit téléchargé sur le site Eumetview.*

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

Le premier fichier TIF proposé est le suivant : 20261007_geocolor_rgb_eumetview.tif </br>
Il se trouve dans ce répertoire : **`/stockage/DATA/eumetview/`**</br></br>
❗ Attention : ce chemin doit-être modifié pour toute arborescence différente

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
INPUT="/stockage/DATA/eumetview/20261007_geocolor_rgb_eumetview.tif"
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
#INPUT="/stockage/DATA/eumetview/20261005_geocolor_rgb_eumetview.tif"
print(f"📁 Fichier source configuré : {INPUT}")

<div class="alert alert-info alert-success">
<h3> 1 - 🔎 Visualisation du TIFF et de son contenu</h3>
</div>

La commande `gdalinfo` affiche les métadonnées géospatiales d’un fichier raster. Dans Jupyter, on la précède de `!`.

In [ ]:
!gdalinfo $INPUT

## 📋 Décryptage des informations clés (fichier mis à jour)

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Driver** | GTiff/GeoTIFF | Format standard géoréférencé (inchangé) |
| 📛 **Fichier** | `20261007_geocolor_rgb_eumetview.tif` | Produit **EUMETVIEW Geocolor**, date **2026-10-07 12:00 UTC** |
| 📐 **Taille** | **11408 × 10857** pixels | **Image rectangulaire** (non carrée, contrairement au disque complet) |
| 🛰️ **Projection** | **GEOGCRS WGS 84 (EPSG:4326)** | **Coordonnées géographiques (Lat/Lon)**. Projection équirectangulaire découpée. |
| 📍 **Étendue géographique** | Lon : -81.27° à 81.28°<br>Lat : -77.35° à 77.35° | Couvre une large bande du globe (découpe rectangulaire). |
| 🧭 **Origine (coin haut-gauche)** | **(-81.2778°, 77.3564°)** | Exprimée en **degrés décimaux** (et non en mètres). |
| 📏 **Pixel Size** | **(0.0142495°, -0.0142495°)** | Résolution d'**~1,6 km/pixel** à l'équateur (car 0.014° × 111 km ≈ 1,58 km). |
| 🏷️ **Métadonnées spécifiques** | `TIFFTAG_DATETIME=2026:10:07 12:00:00` |  |
| 🕒 **Date image** | 2026-10-07 12:00:00 UTC | Heure du slot |
| 🗜️ **Compression / Structure** | **DEFLATE** (sans perte)<br>Interleave = PIXEL | **Changement majeur** : DIfférent par rapport à Météo-France du JPEG/YCbCr. C'est du RGB natif compressé sans perte (qualité maximale). |
| 🎨 **Bandes** | Rouge, Vert, Bleu | 8 bits par canal, **NoData = 0** (zones hors image en noir pur). |
| 🗂️ **Pyramides (Overviews)** | **5704×5429, 2852×2715, 1426×1358, 713×679** | Versions réduites pour l'affichage rapide (adaptées au nouveau format rectangulaire). | 


💡Remarque : la résolution est ici environ affichée à 1,6km en raison de la projection EPSG:4326

📁 **Création du dossier de résultats**

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'

## 🖼️ **Redimensionnement et conversion JPG**

Nous allons redimensionner l'image et la convertir en JPG. L'utilisation de l'overview `[3]` (image 700x700 pixels) permet un gain de temps.

In [ ]:
!convert -quiet -resize 700x700 $INPUT[3]  $output/20261007_geocolor_rgb_eumetview_700x700.jpg

### 🖼️ **Visualisation de l'image**

In [ ]:
im = Image.open(output + '/20261007_geocolor_rgb_eumetview_700x700.jpg', 'r')
display(im)

📌 **Description** : Image **Geocolor RGB** en vue 4326, depuis le satellite **Meteosat-12** positionné à 0° de longitude. Ce produit est réalisé à partir de fichiers chunks, avec diverses corrections (Diffusion, parallaxe, correction angle solaire).

> 💡 *Le produit Geocolor RGB combine plusieurs canaux pour restituer une image en "vraies couleurs" jour et nuit.*

### 🗺️ **Projection de Eckert-IV**

In [ ]:
!gdalwarp -overwrite -s_srs EPSG:4326 -t_srs "+proj=eck4 +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs" -ts 1269 1353 -r cubic $INPUT {output}/20261007_geocolor_rgb_eumetview_eckertIV.tif -co COMPRESS=LZW # >/dev/null 2>&1
!convert -resize 1000 {output}/20261007_geocolor_rgb_eumetview_eckertIV.tif {output}/20261007_geocolor_rgb_eumetview_eckertIV.jpg >/dev/null 2>&1
imEckertIV = Image.open(output + '/20261007_geocolor_rgb_eumetview_eckertIV.jpg', 'r')
display(imEckertIV)

<div class="alert alert-info alert-success">
<h3> 4 - ✂️ Découpes (cropping)</h3>
</div>

### 🇫🇷 **Découpe centrée sur la France (projection EPSG:4326)**

L'avantage de la projection WGS84 est de pouvoir spécifier les limites en degrés.

**Options importantes** :
- `-t_srs {proj}` : nouvelle projection
- `-te {west} {south} {east} {north}` : étendue de découpe (W S E N)
- `-ts {width} {height}` : taille de sortie en pixels
- `-tr {xres} {yres}` : résolution spatiale

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 960 540  $INPUT $output/20261007_geocolor_rgb_eumetview_france.tif #>/dev/null 2>&1
!convert $output/20261007_geocolor_rgb_eumetview_france.tif $output/20261007_geocolor_rgb_eumetview_900_france.jpg #>/dev/null 2>&1
im2 = Image.open(output + '/20261007_geocolor_rgb_eumetview_900_france.jpg', 'r')
display(im2)

> ⚠️ **Attention** : L'image peut être déformée si le ratio de sortie (`-ts`) ne correspond pas au ratio géographique de la zone découpée.</br>
> Ici, la taille 960x540 correspond a du 16/9.

<div class="alert alert-info alert-success">
<h3> 6 - 🖌️ Ajouter un contour</h3>
</div>

### 🗺️ **Ajout de contours avec `gdal_rasterize`**

La commande `gdal_rasterize` permet de dessiner des contours (fichiers shapefile) sur une image.

**Options de couleur** :
- `-b 1 -burn 255 -b 2 -burn 255 -b 3 -burn 255` → blanc
- `-b 1 -burn 0 -b 2 -burn 0 -b 3 -burn 0` → noir
- `-b 1 -burn 255 -b 2 -burn 0 -b 3 -burn 0` → rouge

> ⚠️ *Attention : `gdal_rasterize` modifie directement le fichier d'entrée (le TIF est écrasé).*

In [ ]:
!gdalwarp -overwrite -t_srs EPSG:4326 -te -14 38 18 56 -ts 960 540 -r cubic $INPUT $output/20261007_geocolor_rgb_eumetview_france.tif
!gdal_rasterize -b 1 -burn 200 -b 2 -burn 200 -b 3 -burn 200 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp  $output/20261007_geocolor_rgb_eumetview_france.tif >/dev/null 2>&1
!convert -quiet -resize 1920x1080 $output/20261007_geocolor_rgb_eumetview_france.tif $output/20261007_geocolor_rgb_eumetview_france_contour.jpg
im18 = Image.open(output + '/20261007_geocolor_rgb_eumetview_france_contour.jpg', 'r')
display(im18)

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

A partir de :
  - ce même fichier tiff, ou </br>
  - du fichier tiff Cloud Type (20261007_cloudtype_rgb_eumetview.tif) ou Cloud Phase  (20261007_cloudphase_rgb_eumetview.tif) : </br>
    
-> Découper ce tiff sur la zone de votre choix  : choisir le domaine et la projection, convertir en jpg redimensionné </br>
-> Bonus : ajouter les frontières et un titre sur l'image


## 🌟 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Documentation GDAL](https://gdal.org/en/stable/) | Référence complète de GDAL |

---

✅ **Fin du TP**